## 1. Mechanical Refresher

* A subclass inherits attributes and methods from its parent class, can override a parent method by defining a method with the same name, and can call the parent implementation through `super()`.

* Method lookup checks the instance's class first, then parent classes in method-resolution order, and composition means storing another object and delegating work to it instead of inheriting from it.

## 2. Minimal Working Example

* `TimedReport` gets `name` setup from `Report.__init__` through `super()`, then adds `seconds`.

* Its `label` method overrides the parent method, calls the parent version, and extends the returned string.

In [4]:
class Report:

  def __init__(self, name):
    self.name = name

  def label(self):
    return "report: " + self.name

class TimedReport(Report):

  # self is the TimedReport object
  # name is used by the Report constructor
  # seconds is additional data specific to TimedReport
  def __init__(self, name, seconds):
    super().__init__(name)
    self.seconds = seconds

  def label(self):
    return super().label() + " in " + str(self.seconds) + "s"

print(TimedReport("train", 8).label()) # "train" is passed to Report.__init__ through super(), while 8 is stored as TimedReport-specific data

report: train in 8s


## 3. Modify Drills

**Modify Drill 1.** Change the subclass label suffix and predict the final string.

In [5]:
class Label:

  def __init__(self, instrument):
    self.instrument = instrument

  # Label defines this method, so subclasses such as LoudLabel can inherit it or override it
  def label(self):
    return self.instrument

class LoudLabel(Label):

  def __init__(self, instrument, sound):
    super().__init__(instrument)
    self.sound = sound

  def label(self):
    return super().label() + str(self.sound)

actual = LoudLabel("base", "!").label() # Creates a LoudLabel object, then calls its label() method
expected = "base!"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: base! actual: base! match: True


**Modify Drill 2.** Add a subclass-only attribute through `super().__init__` plus extra assignment.

In [6]:
class Run:

  def __init__(self, name):
    self.name = name

class ScoreRun(Run):

  def __init__(self, name, score):
    super().__init__(name)
    self.score = score

run = ScoreRun("trial", 0.8)
actual = [run.name, run.score]
expected = ["trial", 0.8]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: ['trial', 0.8] actual: ['trial', 0.8] match: True


**Modify Drill 3.** Prefer composition when the new object only uses another object's service.

In [7]:
class Formatter:

  def format(self, value):
    return "[" + str(value) + "]"

class Printer:

  # Stores a Formatter object inside the Printer object
  def __init__(self, formatter):
    self.formatter = formatter

  # Calls the Formatter object's format() method
  def preview(self, value):
    return self.formatter.format(value)

actual = Printer(Formatter()).preview(7)
expected = "[7]"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: [7] actual: [7] match: True


## 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.**

Break it by deleting `super().__init__(name)`. Predict why `self.name` is missing, then restore the parent initialization call.

In [8]:
class Named:
  def __init__(self, name): # Deleting would generate NameError: name 'name' is not defined
    self.name = name

class Tagged(Named):
  def __init__(self, name, tag):
    super().__init__(name)
    self.tag = tag

  def label(self):
    return self.tag + ":" + self.name

actual = Tagged("alpha", "run").label()
expected = "run:alpha"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: run:alpha actual: run:alpha match: True


**Break-and-Fix Drill 2.**

Break it by replacing `super().value()` with `self.value()`. Predict the infinite recursion, then call the parent version with `super()`.

In [9]:
class BaseValue:

  def value(self):
    return 10

class OffsetValue(BaseValue):

  def value(self):
    return super().value() + 5 # Call the value() method defined by parent class BaseValue, not OffsetValue
    # return self.value() + 5 # RecursionError: OffsetValue.value() → self.value() → OffsetValue.value() → ...

actual = OffsetValue().value()
expected = 15
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: 15 actual: 15 match: True


## 5. Self-Verification

Expected-vs-actual prints remain the verification mechanism until the `assert` chapter.

In break-and-fix drills, verify both the fixed value and your prediction of the broken symptom.

## 6. Standalone Exercises

**Exercise 1.** Make `Dog.speak` override `Animal.speak`. Expected behavior: `'woof'`.

In [10]:
class Animal:

  def speak(self):
    return "woof"

class Dog(Animal):

  def speak(self):
    return super().speak()

actual = Dog().speak()
expected = "woof"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: woof actual: woof match: True


**Exercise 2.** Use `super()` so the subclass keeps the parent setup. Expected behavior: `['run', 3]`.

In [11]:
class NamedItem:

    def __init__(self, name):
        self.name = name

class CountedItem(NamedItem):

    def __init__(self, name, count):
        super().__init__(name)
        self.count = count

item = CountedItem("run", 3)
actual = [item.name, item.count]
expected = ["run", 3]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: ['run', 3] actual: ['run', 3] match: True


**Exercise 3.** Override and extend a parent method. Expected behavior: `'value=5; unit=kg'`.

In [12]:
class Measurement:

    def describe(self):
        return "value=5"

class WeightedMeasurement(Measurement):

    def describe(self):
        return super().describe() + "; unit=kg"

actual = WeightedMeasurement().describe()
expected = "value=5; unit=kg"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: value=5; unit=kg actual: value=5; unit=kg match: True


**Exercise 4.** Use composition to reuse `Tokenizer` without inheriting from it. Expected behavior: `['a', 'b']`.

In [13]:
class Tokenizer:

    def split(self,text):
        return text.split("-")

class Parser:

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def parse(self, text):
        return self.tokenizer.split(text)

actual = Parser(Tokenizer()).parse("a-b")
expected = ["a", "b"]
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: ['a', 'b'] actual: ['a', 'b'] match: True


**Exercise 5.** Build a two-level inheritance chain and predict lookup. Expected behavior: `'child-parent-root'`.

In [14]:
class Root:

    def name(self):
        return "root"

class Parent(Root):

    def name(self):
        return "parent-" + super().name()

class Child(Parent):

    def name(self):
        return "child-" + super().name()

actual = Child().name()
expected = "child-parent-root"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: child-parent-root actual: child-parent-root match: True


## 7. Applied AI/ML Drill

**ML to Python mirror:**

a specialized tracker subclass is just an inherited class that reuses parent storage and overrides the part that formats the result.

**Python to ML mirror:**

model and callback libraries use the same pattern when a base class supplies shared bookkeeping and subclasses customize one method such as `step`, `forward`, or `on_epoch_end`.

In [15]:
class MetricTracker:

    def __init__(self, name):
        self.name = name
        self.values = []

    def add(self, value):
        self.values.append(value)

    def summary(self):
        return "tracker:" + self.name

class LossTracker(MetricTracker):

    def summary(self): # self.values is already inherited from MetricTracker because LossTracker did not have def __init__, so default inherited the def __init__ from MetricTracker
        return super().summary() + " last=" + str(self.values[-1])

tracker = LossTracker("loss")
tracker.add(0.25)
actual = tracker.summary()
expected = "tracker:loss last=0.25"
print("expected:", expected, "actual:", actual, "match:", actual == expected)

expected: tracker:loss last=0.25 actual: tracker:loss last=0.25 match: True


## 8. Common Bugs

- **Skipping `super().__init__`**: parent attributes never get created; the symptom is usually `AttributeError` later.

- **Calling `self.method()` inside the overriding method when you meant `super().method()`**: the symptom is infinite recursion.

- **Inheriting only to reuse 1 helper method**: the symptom is a confusing class hierarchy; composition is usually clearer.

- **Assuming a parent method edits subclass state automatically**: `super()` calls exactly the parent method you ask for; it does not guess extra subclass behavior.

## 9. Compounding Drill

Combine inheritance with Chapter 1 function objects: make a subclass that stores a metric function and extends a parent report.

In [17]:
def abs_error(pred, target):
  return abs(pred - target)

def accuracy(pred, target):
    return pred / target

class MetricReport():

  def __init__(self, name, metric_fn):
    self.name = name
    self.metric_fn = metric_fn

  def evaluate(self, pred, target):
    return self.name + '=' + str(self.metric_fn(pred, target))

class SplitMetricReport(MetricReport):

    def __init__(self, name, metric_fn, split):
        super().__init__(name, metric_fn)
        self.split=split

    def evaluate(self, pred, target):
        return self.split + ":" + super().evaluate(pred, target)

# Stretch Goal 1: second subclass for percentage formatting
class PercentageMetricReport(MetricReport):

    def evaluate(self, pred, target):
        result = self.metric_fn(pred, target)
        return self.name + "=" + str(result * 100) + "%"

# Stretch Goal 2: composition-based formatter
class Formatter:

    def format(self, name, value):
        return name + "=" + str(value)

class PercentageFormatter:

    def format(self, name, value):
        return name + "=" + str(value * 100) + "%"

class FormattedMetricReport(MetricReport):

    def __init__(self, name, metric_fn, formatter):
        super().__init__(name, metric_fn)
        self.formatter = formatter

    def evaluate(self, pred, target):
        result = self.metric_fn(pred, target)
        return self.formatter.format(self.name, result)


# Inheritance: split report
split_report = SplitMetricReport("abs_error", abs_error, "train")
actual_1 = split_report.evaluate(5, 2)
expected_1 = "train:abs_error=3"

print("expected:", expected_1, "actual:", actual_1, "match:", actual_1 == expected_1)


# Inheritance: percentage report
percentage_report = PercentageMetricReport("accuracy", accuracy)
actual_2 = percentage_report.evaluate(8, 10)
expected_2 = "accuracy=80.0%"

print("expected:", expected_2, "actual:", actual_2, "match:", actual_2 == expected_2)


# Composition: normal formatter
formatted_report = FormattedMetricReport("abs_error", abs_error, Formatter())
actual_3 = formatted_report.evaluate(5, 2)
expected_3 = "abs_error=3"

print("expected:", expected_3, "actual:", actual_3, "match:", actual_3 == expected_3)


# Composition: percentage formatter
formatted_percentage_report = FormattedMetricReport(
    "accuracy",
    accuracy,
    PercentageFormatter()
)

actual_4 = formatted_percentage_report.evaluate(8, 10)
expected_4 = "accuracy=80.0%"

print("expected:", expected_4, "actual:", actual_4, "match:", actual_4 == expected_4)

expected: train:abs_error=3 actual: train:abs_error=3 match: True
expected: accuracy=80.0% actual: accuracy=80.0% match: True
expected: abs_error=3 actual: abs_error=3 match: True
expected: accuracy=80.0% actual: accuracy=80.0% match: True


## 10. Chapter Project

**Goal:**

Build a tiny metric-reporting system using functions as objects, instance state, inheritance, and `super()`.

**Requirements:**

* create a base `MetricReport` with `name` and `metric_fn`;

* create one subclass that adds a `split` label;

* create an `evaluate(pred, target)` method that returns a readable string.

**Stretch Goals:**

* add a second subclass for percentage formatting;

* add a composition-based formatter object and compare the design.

**Evaluation Checklist:**

* separate instances do not share state; subclass setup calls `super()`;

* the metric function is passed in, not hard-coded;

* expected-vs-actual output confirms at least two metrics.